## BPE Tokenizers (SentencePiece)

### 1 Character Frequency and Coverage Analysis

In [1]:
# imports
from pathlib import Path # filepath tool
from collections import Counter # tool to count characters
import sentencepiece as spm # import SentencePiece library, labeled 'spm'
import json

DATA_DIR = Path("/srv/data/lt2326-h26/a1") # location of data

#### Objective

In [2]:
# Count character occurrences across all three training languages
char_counts = Counter() # dictionary to hold character counts

for lang in ["en", "tr", "zh"]: # iterate over each language dataset
    file_path = DATA_DIR / "train" / f"{lang}.txt" # training dataset

    with open(file_path, "r", encoding="utf-8") as f: # open training dataset in read mode
        for line in f: # for each line in the dataset
            char_counts.update(line.rstrip("\r\n")) # add to character count dictionary (remove return and newline characters)

# Calculate how many characters are needed for each coverage level
total_chars = sum(char_counts.values()) # total number of character counts in character count dictionary

for coverage in [0.90, 0.95, 0.97, 0.99, 0.995, 0.999, 0.9995, 1.0]: # iterate over each degree of coverage listed
    cumulative = 0 # running cumulative frequency counts for each character
    required_chars = 0 # running count of distinct characters

    for char, count in char_counts.most_common(): # for each character and count in the character counts dictionary (descending order)
        cumulative += count # add that character's count to cumulative total
        required_chars += 1 # add 1 to required characters total

        if cumulative / total_chars >= coverage: # if the cumulative occurrences/character occurrneces is greater than/equal to respective coverage 
            break # exit (inner) loop

    print(f"Coverage {coverage:.4f}: {required_chars:,} characters") # print number of required characters for respective coverage

Coverage 0.9000: 457 characters
Coverage 0.9500: 973 characters
Coverage 0.9700: 1,422 characters
Coverage 0.9900: 2,533 characters
Coverage 0.9950: 3,335 characters
Coverage 0.9990: 5,497 characters
Coverage 0.9995: 6,490 characters
Coverage 1.0000: 9,442 characters


### 2 Language Specific Character Coverage

In [3]:
# Select the most frequent characters covering 95% of the combined corpus
coverage_target = 0.95 # set coverage percentage
cumulative = 0 # running cumulative frequency counts for each character
selected_chars = set() # set to hold selected characters (most frequent characters from combined corpus)

for char, count in char_counts.most_common(): # for each character and count in the character counts dictionary (descending order)
    selected_chars.add(char) # add the character to the selected_chars set
    cumulative += count # add one to the cumulative count for character's frequency

    if cumulative / total_chars >= coverage_target: # if the cumulative occurrences/character occurrneces is greater than/equal to the coverage target (0.95)
        break # exit loop

print(f"Characters selected: {len(selected_chars):,}") # print number of selected characters

# Measure how well these characters cover each language

for lang in ["en", "tr", "zh"]: # iterate over each language in dataset

    file_path = DATA_DIR / "train" / f"{lang}.txt" # select training dataset via filepath

    lang_counts = Counter() # dictionary to hold character count for the current language training data

    with open(file_path, "r", encoding="utf-8") as f: # open respective language training dataset in read mode
        for line in f: # iterate over each line in the respective language training dataset
            lang_counts.update(line.rstrip("\r\n")) # removes returns/newlines and adds all remaining line characters to lang_counts dictionary

    total = sum(lang_counts.values()) # total number of character occurrences in current language training data

    covered = sum( # sum of all character occurrences found in the selected character set from the current language training data
        count
        for char, count in lang_counts.items() # iterate over each character and its frequency in the language counts dictionary
        if char in selected_chars # if the character is found in the selected character set
    )

    excluded = total - covered # characters not covered (total characters - covered characters)

    print(f"\nLanguage: {lang}") # print current language
    print(f"Character coverage: {covered / total:.2%}") # percentage of covered characters in current language training data
    print(f"Character occurrences excluded: {excluded:,}") # number of characters occurrences not covered in current language training data
    print(f"Distinct characters excluded: {len(set(lang_counts) - selected_chars):,}") # number of unique characters excluded in current language training data

Characters selected: 973

Language: en
Character coverage: 99.92%
Character occurrences excluded: 2,422
Distinct characters excluded: 493

Language: tr
Character coverage: 99.91%
Character occurrences excluded: 2,519
Distinct characters excluded: 377

Language: zh
Character coverage: 85.17%
Character occurrences excluded: 436,786
Distinct characters excluded: 8,245


#### Comparing Character Coverage Thresholds

In [4]:
# Compare language-specific coverage at different thresholds
coverage_levels = [0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999, 0.9995, 1.0] # list of coverage level percentage thresholds

# Count characters separately for each language
language_counts = {} # dictionary to hold language character counts

for lang in ["en", "tr", "zh"]: # iterate over each language in dataset
    file_path = DATA_DIR / "train" / f"{lang}.txt" # select training dataset via filepath

    counts = Counter() # dictionary to hold character count for the current language training data

    with open(file_path, "r", encoding="utf-8") as f: # open respective language training dataset in read mode
        for line in f: # iterate over each line in the respective language training dataset
            counts.update(line.rstrip("\r\n")) # removes returns/newlines and adds all remaining line characters to Counts dictionary

    language_counts[lang] = counts # add to language_counts dictionary to hold current language character counts

# Calculate coverage for each language at each threshold
for target in coverage_levels: # iterate over each target threshold in coverage_levels list

    cumulative = 0 # running cumulative total of character frequency counts
    selected = set() # set of selected characters

    for char, count in char_counts.most_common(): # iterate over characters and frequency counts in character_counts dictionary
        selected.add(char) # add character to selected character set
        cumulative += count # add character frequency to cumulative frequency count

        if cumulative / total_chars >= target: # if ratio of cumulative character counts divided by total number of character counts is greater than or equal to current target threshold
            break # end inner loop

    print(f"\nOverall coverage target: {target:.0%}") # print current target threshold
    print(f"Characters selected: {len(selected):,}") # print total number of selected characters

    for lang, counts in language_counts.items(): # iterate over language and counts in the language_counts dictionary

        total = sum(counts.values()) # total number of character counts for current language in training corpus

        covered = sum( # sum of all character occurrences found in the selected character set from the current language training data
            count for char, count in counts.items() # iterate over each character and its frequency in the language counts dictionary
            if char in selected # if the character is found in the selected character set
        )

        print(f"  {lang}: {covered / total:.2%}") # print percentage of character occurrences in the current language to 2 decimal points


Overall coverage target: 90%
Characters selected: 457
  en: 99.86%
  tr: 99.73%
  zh: 70.44%

Overall coverage target: 95%
Characters selected: 973
  en: 99.92%
  tr: 99.91%
  zh: 85.17%

Overall coverage target: 97%
Characters selected: 1,422
  en: 99.93%
  tr: 99.93%
  zh: 91.14%

Overall coverage target: 98%
Characters selected: 1,805
  en: 99.94%
  tr: 99.95%
  zh: 94.11%

Overall coverage target: 99%
Characters selected: 2,533
  en: 99.96%
  tr: 99.96%
  zh: 97.08%

Overall coverage target: 100%
Characters selected: 3,335
  en: 99.97%
  tr: 99.97%
  zh: 98.56%

Overall coverage target: 100%
Characters selected: 5,497
  en: 99.99%
  tr: 99.99%
  zh: 99.72%

Overall coverage target: 100%
Characters selected: 6,490
  en: 100.00%
  tr: 100.00%
  zh: 99.86%

Overall coverage target: 100%
Characters selected: 9,442
  en: 100.00%
  tr: 100.00%
  zh: 100.00%


### 3 SentencePiece BPE Training

#### 3.1 Training Corpus Verification

In [5]:
# Locate the prepared multilingual training corpus
TOKENIZER_DATA = DATA_DIR / "tokenizer" / "balanced.txt" # filepath for tokenizer data

print("File exists:", TOKENIZER_DATA.exists()) # return true if tokenizer data file exists (boolean)

if TOKENIZER_DATA.exists(): # if tokenizer data file exists
    print(f"File size: {TOKENIZER_DATA.stat().st_size / 1_000_000:.2f} MB") # print file size to 2 decimal places

    print("\nFirst five lines:") # print label

    with open(TOKENIZER_DATA, "r", encoding="utf-8") as f: # open tokenizer data file in read mode
        for _ in range(5): # iterate over the first 5 lines in tokenizer data file
            print(repr(f.readline()[:200])) # print first five lines up to 200 characters per line including showing invisible characters (such as newline characters)

File exists: True
File size: 14.24 MB

First five lines:
'But French people use the word county (comté) in the name of the Free County region, the old Free County of Burgundy.\n'
'Daha sonra buna ek olarak Angel yayın yapmaya başlamıştır.\n'
'Formül aynı zamanda gözlemlenemeyen morötesi ve kızılötesi ışıkların bazı ek spektral çizgilerini tahmin etmiştir.\n'
"İGD Dış Bürosu tarafından 1980'li yıllarda yayınlanan haber bülteni Türkçenin yanı sıra İngilizce ve Fransızca olarak da çıkmıştır.\n"
'Bu durum sayesinde düşmanlarından gizlenir.\n'


#### 3.2 Model Output Directory

In [6]:
# Directory for saved tokenizer models
TOKENIZER_DIR = Path("../models/tokenizers") # filepath for tokenizer models
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True) # make folder for tokenizer output, including any missing parent directories

print("Tokenizer output directory:", TOKENIZER_DIR.resolve()) # print filepath for tokenizer output

Tokenizer output directory: /home/gussougr@GU.GU.SE/lt2326-assignment1/models/tokenizers


#### 3.3 Training the Small BPE Tokenizer (2,000 Tokens)

In [7]:
# Train the small SentencePiece BPE tokenizer
spm.SentencePieceTrainer.train( # SentencePiece function for training the tokenizer: trainer; hyperparameters below
    input=str(TOKENIZER_DATA), # input = tokenizer data
    model_prefix=str(TOKENIZER_DIR / "bpe_2000"), # output data saved with prefix 'bpe_2000'
    vocab_size=2000, # set vocabulary size at 2000
    model_type="bpe", # use BPE model
    character_coverage=0.98 # set coverage percentage to 98%
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /srv/data/lt2326-h26/a1/tokenizer/balanced.txt
  input_format: 
  model_prefix: ../models/tokenizers/bpe_2000
  model_type: BPE
  vocab_size: 2000
  self_test_sample_size: 0
  character_coverage: 0.98
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surf

#### 3.4 Verification of Small BPE Tokenizer (2000 tokens)

In [8]:
# Load the trained SentencePiece model
bpe_small = spm.SentencePieceProcessor( # create tokenizer object by loading the trained model from above cell
    model_file=str(TOKENIZER_DIR / "bpe_2000.model") # filepath for 2000 vocab model
)

# Verify vocabulary size
print("Vocabulary size:", bpe_small.get_piece_size()) # print vocabulary size of model

# Test encoding and decoding
examples = { # create dictionary of 3 sample sentences, 1 for each language
    "en": "The children are playing outside.",
    "tr": "Çocuklar dışarıda oynuyor.",
    "zh": "孩子们正在外面玩。"
}

for lang, text in examples.items(): # iterate over each language and sentence in the examples dictionary
    tokens = bpe_small.encode(text, out_type=str) # divide sentence from examples dictionary into learned tokens; print as strings
    decoded = bpe_small.decode(tokens) # decode each sentence from tokens

    print(f"\nLanguage: {lang}") # print respective language
    print("Original:", text) # print original text in examples dictionary
    print("Tokens:", tokens) # print encoded sentence (tokens)
    print("Decoded:", decoded) # print decoded sentence

Vocabulary size: 2000

Language: en
Original: The children are playing outside.
Tokens: ['▁The', '▁c', 'h', 'il', 'd', 'r', 'en', '▁are', '▁p', 'l', 'ay', 'ing', '▁o', 'ut', 's', 'i', 'de', '.']
Decoded: The children are playing outside.

Language: tr
Original: Çocuklar dışarıda oynuyor.
Tokens: ['▁', 'Ç', 'o', 'c', 'u', 'k', 'lar', '▁d', 'ış', 'ar', 'ı', 'da', '▁o', 'y', 'n', 'u', 'y', 'or', '.']
Decoded: Çocuklar dışarıda oynuyor.

Language: zh
Original: 孩子们正在外面玩。
Tokens: ['▁', '孩', '子', '们', '正', '在', '外', '面', '玩', '。']
Decoded: 孩子们正在外面玩。


#### 3.5 Training the Large BPE Tokenizer (10,000 Tokens)

In [9]:
# Train the large SentencePiece BPE tokenizer

spm.SentencePieceTrainer.train( # SentencePiece function for training the tokenizer: trainer; hyperparameters below
    input=str(TOKENIZER_DATA), # input = tokenizer data
    model_prefix=str(TOKENIZER_DIR / "bpe_10000"), # output data saved with prefix 'bpe_10000'
    vocab_size=10000, # set vocabulary size at 10000
    model_type="bpe", # use BPE model
    character_coverage=0.98 # set coverage percentage to 98%
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /srv/data/lt2326-h26/a1/tokenizer/balanced.txt
  input_format: 
  model_prefix: ../models/tokenizers/bpe_10000
  model_type: BPE
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.98
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_su

q=129 size=4440 all=698830 active=37171 piece=因素
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=128 size=4460 all=699927 active=38268 piece=98
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=128 size=4480 all=701561 active=39902 piece=omes
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=127 size=4500 all=702304 active=40645 piece=相似
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=127 min_freq=12
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=127 size=4520 all=703092 active=35812 piece=▁read
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=126 size=4540 all=704239 active=36959 piece=日至
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=126 size=4560 all=705285 active=38005 piece=▁düşük
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=125 size=4580 all=706640 active=39360 piece=经常
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=125 size=4600 all=707646 active=40366 piece=▁çoğu
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=125 min_freq=12
bpe_mod

#### 3.4 Verification of Large BPE Tokenizer (10,000 tokens)

In [10]:
# Load the trained large BPE tokenizer
bpe_large = spm.SentencePieceProcessor( # create tokenizer object by loading the trained model from above cell
    model_file=str(TOKENIZER_DIR / "bpe_10000.model") # filepath for 10000 vocab model
)

print("Vocabulary size:", bpe_large.get_piece_size()) # print vocabulary size of model

# Test the same three sentences
for lang, text in examples.items(): # iterate over each language and sentence in the examples dictionary (see above cell)
    tokens = bpe_large.encode(text, out_type=str) # divide sentence from examples dictionary into learned tokens; print as strings
    decoded = bpe_large.decode(tokens) # decode each sentence from tokens

    print(f"\nLanguage: {lang}") # print respective language
    print("Original:", text) # print original text in examples dictionary
    print("Tokens:", tokens) # print encoded sentence (tokens)
    print("Decoded:", decoded) # print decoded sentence

Vocabulary size: 10000

Language: en
Original: The children are playing outside.
Tokens: ['▁The', '▁children', '▁are', '▁playing', '▁outside', '.']
Decoded: The children are playing outside.

Language: tr
Original: Çocuklar dışarıda oynuyor.
Tokens: ['▁Ç', 'oc', 'uk', 'lar', '▁dış', 'arı', 'da', '▁oyn', 'uyor', '.']
Decoded: Çocuklar dışarıda oynuyor.

Language: zh
Original: 孩子们正在外面玩。
Tokens: ['▁', '孩子', '们', '正在', '外', '面', '玩', '。']
Decoded: 孩子们正在外面玩。


#### 3.7 Training High-Coverage BPE (10,000 Tokens, 99.95% Coverage)

In [11]:
# Train an additional BPE tokenizer with higher character coverage
spm.SentencePieceTrainer.train( # SentencePiece function for training the tokenizer: trainer; hyperparameters below
    input=str(TOKENIZER_DATA), # input = tokenizer data
    model_prefix=str(TOKENIZER_DIR / "bpe_10000_high_coverage"), # output data saved with prefix 'bpe_10000_high_coverage'
    vocab_size=10000, # set vocabulary size at 10000
    model_type="bpe", # use BPE model
    character_coverage=0.9995 # set coverage percentage to 99.95%
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /srv/data/lt2326-h26/a1/tokenizer/balanced.txt
  input_format: 
  model_prefix: ../models/tokenizers/bpe_10000_high_coverage
  model_type: BPE
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece

#### 3.8 Verify High-Coverage BPE

In [12]:
# Load the trained high-coverage BPE tokenizer
bpe_high = spm.SentencePieceProcessor( # create tokenizer object by loading the trained model from above cell
    model_file=str(TOKENIZER_DIR / "bpe_10000_high_coverage.model") # filepath for 10000 vocab, high coverage model
)

print("Vocabulary size:", bpe_high.get_piece_size()) # print vocabulary size of model

# Test the same three sentences
for lang, text in examples.items(): # iterate over each language and sentence in the examples dictionary (see above cell)
    tokens = bpe_high.encode(text, out_type=str) # divide sentence from examples dictionary into learned tokens; print as strings
    decoded = bpe_high.decode(tokens) # decode each sentence from tokens

    print(f"\nLanguage: {lang}") # print respective language
    print("Original:", text) # print original text in examples dictionary
    print("Tokens:", tokens) # print encoded sentence (tokens)
    print("Decoded:", decoded) # print decoded sentence

Vocabulary size: 10000

Language: en
Original: The children are playing outside.
Tokens: ['▁The', '▁children', '▁are', '▁play', 'ing', '▁out', 'side', '.']
Decoded: The children are playing outside.

Language: tr
Original: Çocuklar dışarıda oynuyor.
Tokens: ['▁Ç', 'oc', 'uk', 'lar', '▁dış', 'arı', 'da', '▁oyn', 'uy', 'or', '.']
Decoded: Çocuklar dışarıda oynuyor.

Language: zh
Original: 孩子们正在外面玩。
Tokens: ['▁', '孩子', '们', '正在', '外', '面', '玩', '。']
Decoded: 孩子们正在外面玩。


### 4. Tokenizer Evaluation

#### 4.1 Validation Statistics

In [13]:
def evaluate_bpe(tokenizer, file_path): # define function for BPE tokenizer evalation; input parameters include BPE model and filepath
    total_tokens = 0 # cumulative tally of total tokens in data
    total_chars = 0 # cumulative tally of total characters
    total_sentences = 0 # cumulative tally of total sentences
    total_unk = 0 # cumulative tally of total unknown tokens

    with open(file_path, "r", encoding="utf-8") as f: # open filepath in read mode
        for line in f: # iterate over each line in the file
            text = line.rstrip("\r\n") # remove returns and newlines from each line

            if not text: # if empty string
                continue # move to next iteration

            token_ids = tokenizer.encode(text, out_type=int) # divide line into learned tokens; output type: integer

            total_tokens += len(token_ids) # add number of line tokens to total_tokens
            total_chars += len(text) # add number of line characters to total_chars
            total_sentences += 1 # add line to total sentences count
            total_unk += token_ids.count(tokenizer.unk_id()) # add total count of unknown tokens from line to total_unk

    return { # return dictionary
        "vocab_size": tokenizer.get_piece_size(), # vocabulary size of BPE model
        "total_tokens": total_tokens, # total tokens in data
        "avg_tokens_per_sentence": total_tokens / total_sentences, # avg tokens per sentence (total tokens divided by total sentences)
        "chars_per_token": total_chars / total_tokens, # characters per token (total characters divided by total tokens)
        "unk_tokens": total_unk # total unknown tokens
    }

In [14]:
# Evaluate both BPE tokenizers on each language
bpe_models = { # dictionary of BPE models
    "BPE 2000": bpe_small, # small BPE model
    "BPE 10000": bpe_large # large BPE model
}

for model_name, tokenizer in bpe_models.items(): # iterate over bpe_models dictionary

    print(f"\n{model_name}") # print current model name

    for lang in ["en", "tr", "zh"]: # iterate over each language in validation data
        file_path = DATA_DIR / "valid" / f"{lang}.txt" # set file path for current language

        results = evaluate_bpe(tokenizer, file_path) # function call with current BPE model and language as arguments

        print(
            f"{lang}: " # print current language
            f"vocab={results['vocab_size']:,}, " # print function return: vocabulary size of bpe model
            f"tokens={results['total_tokens']:,}, " # print function return: total tokens in data
            f"avg/sentence={results['avg_tokens_per_sentence']:.2f}, " # print function return: average tokens per sentence in data
            f"chars/token={results['chars_per_token']:.2f}, " # print function return: characters per token in data
            f"UNK={results['unk_tokens']:,}" # print function return: total unknown tokens in data
        )


BPE 2000
en: vocab=2,000, tokens=201,806, avg/sentence=44.45, chars/token=1.82, UNK=125
tr: vocab=2,000, tokens=216,188, avg/sentence=57.88, chars/token=1.70, UNK=132
zh: vocab=2,000, tokens=352,207, avg/sentence=36.86, chars/token=1.05, UNK=18,847

BPE 10000
en: vocab=10,000, tokens=111,138, avg/sentence=24.48, chars/token=3.31, UNK=125
tr: vocab=10,000, tokens=109,417, avg/sentence=29.30, chars/token=3.37, UNK=132
zh: vocab=10,000, tokens=279,340, avg/sentence=29.24, chars/token=1.32, UNK=18,847


### 5. Exploratory Comparison: Character Coverage

#### 5.1 Validation Statistics

In [15]:
# Compare the two large BPE tokenizers at different coverage settings
for name, tokenizer in [ # iterate over each large (10,000 vocab) BPE tokenizer model
    ("BPE 10000 - 98%", bpe_large), 
    ("BPE 10000 - 99.95%", bpe_high)
]:
    print(f"\n{name}") # print name of current model
 
    for lang in ["en", "tr", "zh"]: # iterate over each language in validation data
        file_path = DATA_DIR / "valid" / f"{lang}.txt" # set filepath for current language in validation data
        results = evaluate_bpe(tokenizer, file_path) # function call for current model and validation data

        print(
            f"{lang}: " # print current language
            f"tokens={results['total_tokens']:,}, " # print function return: total tokens in data
            f"chars/token={results['chars_per_token']:.2f}, " # print function return: characters per token in data
            f"UNK={results['unk_tokens']:,}" # print function return: total unknown tokens in data
        )


BPE 10000 - 98%
en: tokens=111,138, chars/token=3.31, UNK=125
tr: tokens=109,417, chars/token=3.37, UNK=132
zh: tokens=279,340, chars/token=1.32, UNK=18,847

BPE 10000 - 99.95%
en: tokens=126,921, chars/token=2.90, UNK=26
tr: tokens=126,821, chars/token=2.90, UNK=14
zh: tokens=307,149, chars/token=1.20, UNK=732


### 6. Tokenizer Comparison

#### 6.1 Representative Tokenization Examples

In [16]:
# Load the saved character-level tokenizer vocabulary
with open(TOKENIZER_DIR / "character_vocab.json", "r", encoding="utf-8") as f: # open JSON character vocabulary file in read mode
    char_vocab = json.load(f) # convert JSON file to dictionary

char_to_id = char_vocab["char_to_id"] # retrieve character to ID dictionary from char_vocab dictionary
char_unk_id = char_vocab["unk_id"] # retrieve unknown character ID from char_vocab dictionary

print("Character vocabulary size:", len(char_to_id) + 1) # print total character vocabulary size

Character vocabulary size: 9443


In [17]:
# Compare tokenization of the same sentences across all four tokenizers
for lang, text in examples.items(): # iterate over language and lines in example sentences dictionary

    # Character-level tokenization
    char_tokens = [ # create list of characters from sentence
        char if char in char_to_id else "<UNK>" # if not in vocabulary label unknown
        for char in text # for each character in the sentence
    ]

    # BPE tokenization
    small_tokens = bpe_small.encode(text, out_type=str) # using the small BPE tokenizer, divide sentence from examples dictionary into learned tokens; print as strings
    large_tokens = bpe_large.encode(text, out_type=str) # using the large BPE tokenizer, divide sentence from examples dictionary into learned tokens; print as strings
    high_tokens = bpe_high.encode(text, out_type=str) # using the high coverage BPE tokenizer, divide sentence from examples dictionary into learned tokens; print as strings

    print(f"\nLanguage: {lang}") # print current language
    print("Original:", text) # print original text in examples dictionary

    print(f"\nCharacter ({len(char_tokens)} tokens):") # print number of character tokens in current language for sentence
    print(char_tokens) # print vocabulary characters in sentence for current language

    print(f"\nBPE 2000, 98% coverage ({len(small_tokens)} tokens):") # print label for small BPE tokenizer
    print(small_tokens) # print small BPE model tokens for sentence in current language

    print(f"\nBPE 10000, 98% coverage ({len(large_tokens)} tokens):") # print label for large BPE tokenizer
    print(large_tokens) # print large BPE model tokens for sentence in current language

    print(f"\nBPE 10000, 99.95% coverage ({len(high_tokens)} tokens):") # print label for high coverage BPE tokenizer
    print(high_tokens) # print high coverage BPE model tokens for sentence in current language

    print("-" * 60) # print dividing line for visibility


Language: en
Original: The children are playing outside.

Character (33 tokens):
['T', 'h', 'e', ' ', 'c', 'h', 'i', 'l', 'd', 'r', 'e', 'n', ' ', 'a', 'r', 'e', ' ', 'p', 'l', 'a', 'y', 'i', 'n', 'g', ' ', 'o', 'u', 't', 's', 'i', 'd', 'e', '.']

BPE 2000, 98% coverage (18 tokens):
['▁The', '▁c', 'h', 'il', 'd', 'r', 'en', '▁are', '▁p', 'l', 'ay', 'ing', '▁o', 'ut', 's', 'i', 'de', '.']

BPE 10000, 98% coverage (6 tokens):
['▁The', '▁children', '▁are', '▁playing', '▁outside', '.']

BPE 10000, 99.95% coverage (8 tokens):
['▁The', '▁children', '▁are', '▁play', 'ing', '▁out', 'side', '.']
------------------------------------------------------------

Language: tr
Original: Çocuklar dışarıda oynuyor.

Character (26 tokens):
['Ç', 'o', 'c', 'u', 'k', 'l', 'a', 'r', ' ', 'd', 'ı', 'ş', 'a', 'r', 'ı', 'd', 'a', ' ', 'o', 'y', 'n', 'u', 'y', 'o', 'r', '.']

BPE 2000, 98% coverage (19 tokens):
['▁', 'Ç', 'o', 'c', 'u', 'k', 'lar', '▁d', 'ış', 'ar', 'ı', 'da', '▁o', 'y', 'n', 'u', 'y', 'or', '.